# Model recovery and parameter recovery — RL vs RLWM (PyBADS)

This notebook fits **both agents to all datasets generated by both agents**, compares the models with **BIC**, plots a **model-recovery confusion matrix**, and checks **parameter recovery**. Finally, it fits the original `df_rlwm_clean.csv` participants.

**Dependencies:** `numpy`, `pandas`, `matplotlib`, `pybads`, and `joblib>=1.3` (**no `numba` or `tqdm`**). If needed, run `pip install pybads "joblib>=1.3"` in your environment. [PyBADS documentation](https://acerbilab.github.io/pybads/).

**Important: exact models used to generate these CSVs**
- **RL** has `alphaRL` and `betaRL`, with no forgetting.
- **RLWM** adds `decayRL`, `decayWM`, and `WMweight`. WM learns with rate **1**, and both value tables decay **before** the chosen action is updated.
- **WM capacity is fixed: `K = 3`**, *not* fitted. The choice weight in a set of size `ns` is `WMweight * min(1, K/ns)`.
- Every set starts with zero action values. The observed feedback uses **0 for an incorrect action** and **1 or 2 for a correct action**. Actions in the CSV are 1–3 and are converted to 0–2 for fitting.

**Workflow:** first run with `QUICK_TEST = True`, then set it to `False` for all 100 simulations and all real participants. Saved fits resume automatically after interruption. Results are stored separately for quick vs full analyses.

In [1]:
# Settings and imports
from pathlib import Path
import math
import zlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed, parallel_config

QUICK_TEST = False        # Set to False for the complete model/parameter recovery
K = 3                    # Fixed WM capacity used by the uploaded RLWM simulations
N_STARTS = 10             # Independent PyBADS initializations; increase for more robust fits
MAX_FUN_EVALS = 500      # PyBADS function evaluations per start; increase for final analyses
N_JOBS = -1              # Use all available CPUs; set to a positive number to limit workers
BASE_SEED = 12345        # Use to reproduce the same fitting results each time you start this notebook

OUTPUT = Path('recovery_results') / ('quick' if QUICK_TEST else 'full') / f'K{K}'
OUTPUT.mkdir(parents=True, exist_ok=True)

OUTPUT_REALDATA = Path('real_results') / f'K{K}'
OUTPUT_REALDATA.mkdir(parents=True, exist_ok=True)

print('Saving results in:', OUTPUT.resolve())

Saving results in: C:\RobustCircuit\Neurograd26\Practicals\Practical3\recovery_results\full\K3


## 1. Load the datasets

In [2]:
# Paths of the relevant files
sim_path = Path.cwd() / 'simulated_RL_and_RLWM.csv'
rl_path = Path.cwd() / 'parameters_RL.csv'
rlwm_path = Path.cwd() / 'parameters_RLWM.csv'
realdata_path = Path.cwd() / 'preprocessed_data' / 'df_rlwm_clean.csv'
#
df_sim = pd.read_csv(sim_path)
df_rl_params = pd.read_csv(rl_path)
df_rlwm_params = pd.read_csv(rlwm_path)
df_real = pd.read_csv(realdata_path)

# Only learning trials contribute to likelihood fits here.
df_sim = df_sim[df_sim['trial_mode'] == 'learning'].copy()
df_real = df_real[df_real['trial_mode'] == 'learning'].copy()
true_params = pd.concat([df_rl_params, df_rlwm_params], ignore_index=True)

# Check that the simulation parameters & surrogate data files correspond
assert set(df_sim['simId'].unique()) == set(true_params['simId'].unique())

print('Simulations:', df_sim.groupby('model')['simId'].nunique().to_dict())
print('Trials per simulation:', df_sim.groupby('simId').size().unique().tolist())
print('Real participants:', df_real['subjid'].nunique())
print('Real learning trials:', len(df_real))

Simulations: {'RL': 50, 'RLWM': 50}
Trials per simulation: [624]
Real participants: 69
Real learning trials: 25931


## 2. Prepare the choice sequences

Each new `(blockId, setId)` starts a new set of values. **Stimulus IDs in the real data are arbitrary** (for example, a set of size 2 can use `stimId=3` and `5`); they must be remapped to `0,1,...` **within each set**. This is important for fitting the original dataset. Rewards and choices remain exactly as observed.

In [3]:
# Function to recode stimId in real data (fitting functions expected 0,1,...)
def prepare_trials(df):
    df = df.sort_values('trial_index', kind='stable').reset_index(drop=True)
    episodes = []
    for (_, _), block in df.groupby(['blockId', 'setId'], sort=False):
        ns = int(block['setSize'].iloc[0])
        assert (block['setSize'] == ns).all()
        # Remap arbitrary image labels within each set; no values carry over to the next set.
        stim = pd.factorize(block['stimId'], sort=False)[0].astype(int)
        action = block['key_response'].to_numpy(dtype=int) - 1
        reward = block['feedback'].to_numpy(dtype=float)
        assert 1 <= ns <= 6
        assert len(np.unique(stim)) <= ns
        assert np.all((0 <= action) & (action < 3))
        assert np.all((0 <= reward) & (reward <= 2))
        episodes.append((stim, action, reward, ns))
    return episodes

sim_data = {str(sid): prepare_trials(g) for sid, g in df_sim.groupby('simId', sort=False)}
real_data = {str(sid): prepare_trials(g) for sid, g in df_real.groupby('subjid', sort=False)}

print('Prepared:', len(sim_data), 'simulations and', len(real_data), 'participants')
print('Example synthetic set sizes:', [x[3] for x in sim_data['RLWM_001']])
print('Real participant set sizes:', [x[3] for x in next(iter(real_data.values()))])

Prepared: 100 simulations and 69 participants
Example synthetic set sizes: [6, 4, 6, 4, 4, 3, 5, 2, 2, 3, 3, 5, 5]
Real participant set sizes: [2, 2, 3, 5, 4, 2, 3, 5, 3, 5, 4]


## 3. Negative log likelihood for RL and RLWM

We evaluate the probability of **each observed choice before observing its reward**, then update using the actual `feedback`. The objective is deterministic, so it can be passed directly to PyBADS. `replay_seed` is only for an optional validation test, **not** for fitting.

In [4]:
# Parameter order and bounds follow the simulation agents.
PARAM_NAMES = {
    'RL': ['alphaRL', 'betaRL'],
    'RLWM': ['alphaRL', 'betaRL', 'decayRL', 'decayWM', 'WMweight'],
}
BOUNDS = {
    'alphaRL': (0.0001, 1.0),
    'betaRL': (-10.0, 100.0),
    'decayRL': (0.0, 1.0),
    'decayWM': (0.0, 1.0),
    'WMweight': (0.0, 1.0),
}

def softmax3(values, beta):
    # Stable softmax for the 3 possible actions (no overflow even for high beta).
    z0, z1, z2 = beta * values[0], beta * values[1], beta * values[2]
    m = max(z0, z1, z2)
    e0, e1, e2 = math.exp(z0 - m), math.exp(z1 - m), math.exp(z2 - m)
    total = e0 + e1 + e2
    return (e0 / total, e1 / total, e2 / total)

# When we fit data, we need to run MANY times the function that calculates the likelihood of the data
# knowing the model and the parameters (parameters change for each iteration of the fitting routine)
# So, reusing the simulating agents can be quite long if you fit on your own laptop
def neg_log_likelihood(parameters, episodes, model, capacity=K):
    p = np.asarray(parameters, dtype=float).ravel()
    if model == 'RLWM':
        alphaRL, betaRL, decayRL, decayWM, rho = p
    elif model == 'RL':
        alphaRL, betaRL = p[:2]
    else:
        raise ValueError('Unknown model: ' + model)

    nll = 0.0

    for stim, actions, rewards, ns in episodes:
        QRL = np.zeros((ns, 3))
        if model == 'RLWM':
            QWM = np.zeros((ns, 3))
            wm_weight = rho * min(1.0, capacity / ns)

        for s, action, reward in zip(stim, actions, rewards):
            probsRL = softmax3(QRL[s], betaRL)
            if model == 'RLWM':
                probsWM = softmax3(QWM[s], betaRL)
                probs = tuple((1.0 - wm_weight) * probsRL[a] + wm_weight * probsWM[a]
                              for a in range(3))
            else:
                probs = probsRL

            nll -= math.log(max(probs[action], 1e-300))

            # Forgetting across all entries occurs BEFORE updating the chosen action.
            if model == 'RLWM':
                QRL *= (1.0 - decayRL)
                QWM *= (1.0 - decayWM)
            QRL[s, action] += alphaRL * (reward - QRL[s, action])
            if model == 'RLWM':
                QWM[s, action] = reward    # WM learning rate is fixed at 1

    return float(nll)

## 4. Fit with PyBADS

PyBADS minimizes negative log likelihood. For each fit we try `N_STARTS` independent starting points, keep the best, and calculate:

$$
\mathrm{AIC} = 2\,\mathrm{NLL} + 2k,
\qquad
\mathrm{BIC} = 2\,\mathrm{NLL} + k\log(n).
$$

Here `k=2` (RL) or `k=5` (RLWM). **Fixed** capacity `K=3` is not estimated and does not add another fitted parameter. The small `QUICK_TEST` is for checking the workflow; use `QUICK_TEST = False` for the actual recovery analysis.

Note that it is easier to use a partial/anonymous function to perform the fit, so that only model parameters as the only changing input across iterations

In [2]:
# Install once in your own Python environment if necessary:  %pip install pybads "joblib>=1.3"
from pybads import BADS


def fit_one_dataset(episodes, model, seed):
    names = PARAM_NAMES[model]
    lower = np.array([BOUNDS[n][0] for n in names], dtype=float)
    upper = np.array([BOUNDS[n][1] for n in names], dtype=float)
    plausible_lower = lower + 0.01 * (upper - lower)
    plausible_upper = upper - 0.01 * (upper - lower)
    rng = np.random.default_rng(seed)
    objective = lambda x: neg_log_likelihood(x, episodes, model)
    best_nll = np.inf
    best_params = None

    for start in range(N_STARTS):
        x0 = rng.uniform(plausible_lower, plausible_upper)
        bads = BADS(objective, x0, lower, upper, plausible_lower, plausible_upper,
                    options={'display': 'off', 'random_seed': int(seed + 17 * start),
                             'max_fun_evals': MAX_FUN_EVALS})
        result = bads.optimize()
        x_fit = np.asarray(result['x'], dtype=float).ravel()
        nll = objective(x_fit)             # Evaluate fitted parameters directly
        if nll < best_nll:
            best_nll, best_params = nll, x_fit.copy()

    n = sum(len(e[1]) for e in episodes)
    k = len(names)
    row = {'fit_model': model, 'NLL': best_nll, 'AIC': 2 * best_nll + 2 * k,
           'BIC': 2 * best_nll + k * np.log(n), 'n_trials': n,
           'n_parameters': k, 'fixed_K': K if model == 'RLWM' else np.nan}
    for name, value in zip(names, best_params):
        row['fit_' + name] = value
    return row


def _fit_job(job):
    identifier, source_model, model, episodes = job
    text = f'{identifier}-{model}'.encode()
    seed = BASE_SEED + zlib.crc32(text) % 100000
    row = fit_one_dataset(episodes, model, seed)
    row.update({'id': str(identifier), 'source_model': source_model})
    return row


def fit_and_save(jobs, csv_path):
    # Save after each returned fit; rerunning skips completed (id, fit_model) pairs.
    if csv_path.exists():
        fits = pd.read_csv(csv_path)
        done = set(zip(fits['id'].astype(str), fits['fit_model']))
    else:
        fits = pd.DataFrame()
        done = set()

    pending = [(i, job) for i, job in enumerate(jobs, start=1)
               if (str(job[0]), job[2]) not in done]
    if not pending:
        return fits

    # Independent dataset/model fits run in separate processes (Jupyter-compatible).
    # Keep result order and write CSVs only in the main process.
    # Limit numerical-library threads per worker to avoid CPU oversubscription.
    with parallel_config(backend='loky', inner_max_num_threads=1):
        with Parallel(n_jobs=N_JOBS, return_as='generator', batch_size=1) as parallel:
            results = parallel(delayed(_fit_job)(job) for _, job in pending)
            for (i, _), row in zip(pending, results):
                fits = pd.concat([fits, pd.DataFrame([row])], ignore_index=True)
                fits.to_csv(csv_path, index=False)
                if i % 10 == 0 or i == len(jobs):
                    print(f'Completed/checked {i}/{len(jobs)} fits')
    return fits

## 5. Model recovery: fit *both* models to each simulated dataset

For every simulated participant, fit the RL agent and the RLWM agent. A successful model-recovery analysis should select the generating model frequently, **after penalizing extra RLWM parameters with BIC**.

In [6]:
sim_ids = (df_sim[['simId', 'model']].drop_duplicates()
           .sort_values(['model', 'simId']))
if QUICK_TEST:
    sim_ids = sim_ids.groupby('model', group_keys=False).head(2)

sim_jobs = []
for row in sim_ids.itertuples(index=False):
    for candidate_model in ('RL', 'RLWM'):
        sim_jobs.append((row.simId, row.model, candidate_model, sim_data[row.simId]))

print('Running/resuming', len(sim_jobs), 'simulation fits')
sim_fits = fit_and_save(sim_jobs, OUTPUT / 'fits_simulations.csv')
print(sim_fits.groupby(['source_model', 'fit_model']).size())

Running/resuming 200 simulation fits
Completed/checked 10/200 fits
Completed/checked 20/200 fits
Completed/checked 30/200 fits
Completed/checked 40/200 fits
Completed/checked 50/200 fits
Completed/checked 60/200 fits
Completed/checked 70/200 fits
Completed/checked 80/200 fits
Completed/checked 90/200 fits
Completed/checked 100/200 fits
Completed/checked 110/200 fits
Completed/checked 120/200 fits
Completed/checked 130/200 fits
Completed/checked 140/200 fits
Completed/checked 150/200 fits
Completed/checked 160/200 fits
Completed/checked 170/200 fits
Completed/checked 180/200 fits
Completed/checked 190/200 fits
Completed/checked 200/200 fits
source_model  fit_model
RL            RL           50
              RLWM         50
RLWM          RL           50
              RLWM         50
dtype: int64


## 5. Exercise. Fit the original `df_rlwm_clean.csv`

Fit each participant's **learning trials only**, using the same two candidate models (RL and capacity-limited RLWM, fixed `K=3`). A participant's values reset at every stimulus set. Original test-trial responses and participant characteristics are **not** used to fit the learning policy.

Reuse the same functions as before. Try to book-keep the fitting results in a clean way!

**PyBADS reference:** [PyBADS documentation](https://acerbilab.github.io/pybads/) and [BADS API](https://acerbilab.org/pybads/api/classes/bads.html).